In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import display

HOMEWORK_ROOT = Path.cwd()
if str(HOMEWORK_ROOT) not in sys.path:
    sys.path.insert(0, str(HOMEWORK_ROOT))


# Classical Control Homework

This homework contains three required control problems. Each problem includes an interactive viewer for intuition and a small Python environment for testing your controller.

| Problem | Points | Submit |
|---|---:|---|
| A-star-ship: rocket thrust vector control | 10 | `solutions/rocket.py` |
| Vac-Man: robot vacuum planning and control | 20 + 5 leaderboard bonus | `solutions/vacman.py` |
| As easy as riding a bike: unicycle balance and navigation | 20 | `solutions/unicycle.py` |

Run the open tests from this directory with:

```bash
pytest tests/ -v
```

The visualizers help you build intuition; the tests grade the Python controllers.


# Problem 1: A-star-ship (10 pts)
Reusable rocket stages from SpaceX, Blue Origin, and other companies have made controlled takeoff and landing look almost routine. The Artemis launch is another reminder that spaceflight is a major achievement for science, engineering, and robotics.

<div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 12px; margin: 14px 0 18px;">
  <figure style="margin: 0;">
    <video controls muted loop playsinline preload="metadata" style="width: 100%; max-height: 360px; background: #05070a; border: 1px solid #27313d;" src="assets/notebook_cutsies/artemis_takeoff.webm" type="video/webm">
    </video>
    <figcaption style="font-family: monospace; font-size: 12px; color: #9fb2c8; margin-top: 4px;">Artemis launch: controlled ascent.</figcaption>
  </figure>
  <figure style="margin: 0;">
    <video controls muted loop playsinline preload="metadata" style="width: 100%; max-height: 360px; background: #05070a; border: 1px solid #27313d;" src="assets/notebook_cutsies/blue_origin_landing.webm" type="video/webm">
    </video>
    <figcaption style="font-family: monospace; font-size: 12px; color: #9fb2c8; margin-top: 4px;">Reusable booster landing: attitude and vertical-rate control.</figcaption>
  </figure>
</div>

Reusable rocket stages look graceful only when the controller is doing a lot of quiet work. In this problem, you control a simplified 2D rocket with thrust vector control.

State:

$$\mathbf{x} = [x, z, v_x, v_z, \theta, \omega]$$

Control:

$$\mathbf{u} = [T, \delta]$$

where \(T\) is thrust magnitude and \(\delta\) is the engine gimbal angle.

Dynamics:

$$
\dot x = v_x, \qquad \dot z = v_z
$$

$$
\dot v_x = \frac{T}{m}\sin(\theta + \delta) + a_w(t), \qquad
\dot v_z = \frac{T}{m}\cos(\theta + \delta) - g
$$

$$
\dot\theta = \omega, \qquad
\dot\omega = -\frac{T l}{I}\sin(\delta)
$$

| Parameter | Symbol | Value |
|---|---:|---:|
| Mass | \(m\) | 50 kg |
| Inertia | \(I\) | 100 kg m^2 |
| Gimbal lever arm | \(l\) | 1.5 m |
| Max thrust | \(T_{max}\) | 981 N |
| Min thrust | \(T_{min}\) | 0 N |
| Max gimbal | \(\delta_{max}\) | 0.26 rad |

The observation passed to your controller is:

`[target_x, target_z, target_theta, x, z, theta, vx, vz, omega]`

Return:

`[thrust, gimbal_angle]`

Your controller must succeed in both fixed scenarios: ascent and landing. Wind is hidden from the observation.


In [ ]:
from lib.rocket import show_rocket_viewer

show_rocket_viewer(initial_scenario="ascent", wind_enabled=True)


**Task.** Implement `RocketController` in `solutions/rocket.py`.

Minimal hints:

- Stabilize attitude before asking the rocket to do precise translation.
- Treat vertical speed and thrust separately from angular stabilization.
- Tune without wind first, then turn wind back on.


In [ ]:
from lib.rocket import RocketEnv, RocketEnvConfig, WindConfig, make_ascent_scenario, make_landing_scenario
from solutions.rocket import RocketController


def run_rocket_self_check():
    controller = RocketController()
    cases = [
        ("ascent-default", RocketEnvConfig(scenario=make_ascent_scenario(), wind=WindConfig(seed=98765))),
        (
            "ascent-altitude-angle",
            RocketEnvConfig(
                scenario=make_ascent_scenario(z_target=120.0, target_theta=0.10),
                wind=WindConfig(seed=24680),
            ),
        ),
        ("landing-default", RocketEnvConfig(scenario=make_landing_scenario(), wind=WindConfig(seed=98765))),
        (
            "landing-shifted-pad",
            RocketEnvConfig(
                scenario=make_landing_scenario(target_x=14.0),
                wind=WindConfig(seed=24680),
            ),
        ),
    ]
    results = []
    for label, config in cases:
        env = RocketEnv(config)
        if hasattr(controller, "reset"):
            controller.reset()
        obs, info = env.reset()
        max_steps = int(np.ceil(env.scenario.time_limit / env.config.dt))
        reward = 0.0
        for _ in range(max_steps):
            try:
                action = np.asarray(controller(obs), dtype=float)
            except NotImplementedError:
                print("RocketController.__call__ is not implemented yet.")
                return results
            obs, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                break
        ok = info["status"] == "success" and reward == 1.0
        results.append((label, ok, info["status"], float(info["t"])))
        print(f"{label:22s} | ok={ok} | status={info['status']} | t={info['t']:.2f}s")
    return results

rocket_results = run_rocket_self_check()


# Problem 2: Vac-Man (20 pts + 5 leaderboard bonus)

You control a small robot vacuum in a room with dust, walls, limited battery, and a chasing Cat-Man. The goal is to leave the base, clean as much dust as you can, and return to the glowing green base before the battery runs out or Cat-Man catches you.

<div style="margin: 12px 0 18px;">
  <video controls muted loop playsinline preload="metadata" style="width: 100%; max-height: 420px; background: #05070a; border: 1px solid #27313d;" src="assets/notebook_cutsies/vacman_chase.webm" type="video/webm">
  </video>
  <div style="font-family: monospace; font-size: 12px; color: #9fb2c8; margin-top: 4px;">Vac-Man run: almost beat up my record.</div>
</div>

Vac-Man uses differential drive. Your action is:

```text
[v_left, v_right]
```

The environment observation is a dictionary with world state, map data, dust grid, battery, and flags. Useful keys include `vacman`, `catman`, `base`, `dust`, `obstacles`, `cat_cspace_obstacles`, `arena`, `battery`, and `flags`.

Cat-Man behavior is deterministic. Once Vac-Man moves, Cat-Man expands walls by its octagonal footprint (Minkowski sum), builds a visibility graph, runs A* to the current chase target, and moves along the shortest visible path.

A successful episode ends when Vac-Man has left the base and returned to it. The reward equals the amount of dust cleaned. Being caught or running out of charge causes failure. But don't focus much on the charge, it just limits how long the run can be.

In [ ]:
from lib.vacman import show_vacman_viewer

show_vacman_viewer(case_index=1)


**Task.** Implement `VacmanController` in `solutions/vacman.py` which beats human baseline!

Expected controller contract:

```python
class VacmanController:
    def reset(self) -> None: ...
    def __call__(self, obs) -> np.ndarray: ...  # [v_left, v_right]
```

Minimal hints:

- Dust is discretized
- Cat-man Behaviour is deterministic and he uses graph search
- But the env is non-markovian, because you eat dust and this retains across the whole history


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from lib.vacman import VacmanEnv, VacmanEnvConfig, fixed_vacman_cases

try:
    from solutions.vacman import VacmanController
except ModuleNotFoundError:
    VacmanController = None


def run_vacman_self_check(case_id="o_small", max_steps=1600):
    if VacmanController is None:
        print("Expected solutions/vacman.py with class VacmanController.")
        env = VacmanEnv(VacmanEnvConfig(case_id=case_id, debug=True))
        env.reset()
        fig = env.render("matplotlib")
        display(fig)
        plt.close(fig)
        return None

    env = VacmanEnv(VacmanEnvConfig(case_id=case_id, debug=True))
    controller = VacmanController()
    if hasattr(controller, "reset"):
        controller.reset()
    obs, info = env.reset()
    reward = 0.0
    terminated = truncated = False
    for _ in range(max_steps):
        try:
            action = np.asarray(controller(obs), dtype=float)
        except NotImplementedError:
            print("VacmanController.__call__ is not implemented yet.")
            break
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break
    print(
        f"case={info['case_id']} status={info['status']} "
        f"reward={reward:.3f} dust={info['cleaned']}/{info['total_dust']} "
        f"charge={info['battery_remaining']:.1f}/{info['battery_capacity']:.1f}"
    )
    fig = env.render("matplotlib")
    display(fig)
    plt.close(fig)
    return info

print("Available open cases:", fixed_vacman_cases())
vacman_info = run_vacman_self_check("o_small")


# Problem 3: As easy as riding a bike (20 pts)

A humanoid on a unicycle is a less forgiving version of the cart-pole idea. The model here is still simplified, but the controller must balance and navigate at the same time.

The observation is a flat array:

```text
[target_x, target_y, qpos(10), qvel(9)]
```

Expanded order:

```text
[target_x, target_y,
 wheel_x, wheel_y, wheel_z,
 wheel_qw, wheel_qx, wheel_qy, wheel_qz,
 wheel_rotation_angle, pelvis_y_angle, pelvis_x_angle,
 wheel_vx, wheel_vy, wheel_vz,
 wheel_omega_x, wheel_omega_y, wheel_omega_z,
 wheel_angular_velocity, pelvis_y_angular_velocity, pelvis_x_angular_velocity]
```

Return motor torques:

```text
[tau_pelvis_y, tau_pelvis_x, tau_wheel]
```

The fixed tests place the target at deterministic locations. Success means reaching the target with the rider still upright.


In [ ]:
from lib.unicycle import show_unicycle_viewer

show_unicycle_viewer()


**Task.** Implement `UnicycleController` in `solutions/unicycle.py`.

Minimal hints:

- Try RL too, I could not make classical control work reliably. Best of luck <3


In [ ]:
from lib.unicycle import FIXED_TARGET_CASES, UnicycleEnv, UnicycleEnvConfig
from solutions.unicycle import UnicycleController


def run_unicycle_self_check():
    controller = UnicycleController()
    results = []
    for target_xy in FIXED_TARGET_CASES:
        env = UnicycleEnv(UnicycleEnvConfig(target_xy=target_xy))
        if hasattr(controller, "reset"):
            controller.reset()
        obs, info = env.reset(options={"target_xy": target_xy})
        max_steps = int(np.ceil(env.config.time_limit / env.policy_dt))
        reward = 0.0
        for _ in range(max_steps):
            try:
                action = np.asarray(controller(obs), dtype=float)
            except NotImplementedError:
                print("UnicycleController.__call__ is not implemented yet.")
                return results
            obs, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                break
        ok = info["status"] == "success" and reward == 1.0
        results.append((target_xy, ok, info["status"], float(info["distance_to_target"])))
        print(
            f"target={tuple(target_xy)} | ok={ok} | status={info['status']} "
            f"| distance={info['distance_to_target']:.3f}"
        )
    return results

unicycle_results = run_unicycle_self_check()
